# Web scraping Falcon 9 launch records

BeautifulSoup locates launch tables in the fixed Wikipedia revision. pandas reads their cells. We remove description rows and footnotes, retain Falcon 9, parse dates and masses, and preserve launch versus landing outcomes separately. The source HTML is stored for repeatability.

Source: https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922

In [1]:
"""Reproducible SpaceX capstone: scraping, SQL, visual EDA and model evaluation."""
from pathlib import Path
from io import StringIO
import calendar
import json
import re
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, TimeSeriesSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, confusion_matrix


ROOT = Path.cwd().parent
DATA = ROOT / 'data'
FIG = ROOT / 'figures'


In [2]:
def clean_text(value):
    return re.sub(r"\[[^\]]*\]", "", str(value)).replace("\u00a0", " ").strip()

def scrape_wikipedia():
    """Parse historical HTML tables, skipping mission-description rows and Falcon Heavy."""
    source = DATA / "wikipedia_launches.html"
    soup = BeautifulSoup(source.read_text(encoding="utf-8"), "html.parser")
    frames = []
    for table in soup.find_all("table", class_="wikitable"):
        headers = [h.get_text(" ", strip=True) for h in table.find_all("th")[:10]]
        if not headers or headers[0] != "Flight No." or "Booster landing" not in headers:
            continue
        table_df = pd.read_html(StringIO(str(table)))[0]
        if table_df.shape[1] != 10:
            continue
        table_df.columns = ["FlightNumber", "DateText", "BoosterVersion", "LaunchSite", "Payload", "PayloadMassText", "Orbit", "Customer", "LaunchOutcome", "LandingOutcome"]
        frames.append(table_df)
    df = pd.concat(frames, ignore_index=True)
    for col in df.columns:
        df[col] = df[col].map(clean_text)
    dates = df.DateText.str.extract(r"^(\d{1,2} [A-Za-z]+ \d{4})", expand=False)
    df = df[dates.notna() & df.BoosterVersion.str.startswith("F9")].copy()
    df["Date"] = pd.to_datetime(dates.loc[df.index], format="%d %B %Y")
    df["FlightNumber"] = pd.to_numeric(df.FlightNumber.str.extract(r"^(\d+)", expand=False))
    masses = df.PayloadMassText.str.extract(r"([\d,]+(?:\.\d+)?)\s*kg", expand=False)
    df["PayloadMass"] = pd.to_numeric(masses.str.replace(",", "", regex=False), errors="coerce")
    df["LaunchSite"] = df.LaunchSite.str.replace(",", "", regex=False).str.replace("SLC-", "SLC ", regex=False).str.replace("LC-", "LC ", regex=False)
    df["LaunchSuccess"] = df.LaunchOutcome.str.startswith("Success").astype(int)
    df["LandingSuccess"] = df.LandingOutcome.str.startswith("Success").astype(int)
    df["LandingType"] = np.select([df.LandingOutcome.str.contains("drone ship", case=False), df.LandingOutcome.str.contains("ground pad", case=False), df.LandingOutcome.str.contains("ocean", case=False)], ["Drone ship", "Ground pad", "Ocean"], default="Other / no attempt")
    df["LandingResult"] = np.select([df.LandingOutcome.str.startswith("Success"), df.LandingOutcome.str.startswith("Failure"), df.LandingOutcome.str.contains("No attempt", case=False)], ["Success", "Failure", "No attempt"], default="Other")
    df = df.drop_duplicates("FlightNumber").sort_values("Date").reset_index(drop=True)
    df["Date"] = df.Date.dt.strftime("%Y-%m-%d")
    df.to_csv(DATA / "wikipedia_scraped.csv", index=False)
    return df

In [3]:
scraped = scrape_wikipedia()
print('Rows:',len(scraped),'Missing masses:',scraped.PayloadMass.isna().sum())
display(scraped.head())
display(scraped.tail())

Rows: 122 Missing masses: 5


,FlightNumber,DateText,BoosterVersion,LaunchSite,Payload,PayloadMassText,Orbit,Customer,LaunchOutcome,LandingOutcome,Date,PayloadMass,LaunchSuccess,LandingSuccess,LandingType,LandingResult
0,1.0,"4 June 2010, 18:45",F9 v1.0 B0003.1,CCAFS SLC 40,Dragon Spacecraft Qualification Unit,nan,LEO,SpaceX,Success,Failure (parachute),2010-06-04,NaN,1,0,Other / no attempt,Failure
1,2.0,"8 December 2010, 15:43",F9 v1.0 B0004.1,CCAFS SLC 40,Dragon demo flight C1 (Dragon C101),nan,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute),2010-12-08,NaN,1,0,Other / no attempt,Failure
2,3.0,"22 May 2012, 07:44",F9 v1.0 B0005.1,CCAFS SLC 40,Dragon demo flight C2+ (Dragon C102),"525 kg (1,157 lb)",LEO (ISS),NASA (COTS),Success,No attempt,2012-05-22,525.0,1,0,Other / no attempt,No attempt
3,4.0,"8 October 2012, 00:35",F9 v1.0 B0006.1,CCAFS SLC 40,SpaceX CRS-1 (Dragon C103),"4,700 kg (10,400 lb)",LEO (ISS),NASA (CRS),Success,No attempt,2012-10-08,4700.0,1,0,Other / no attempt,No attempt
4,5.0,"1 March 2013, 15:10",F9 v1.0 B0007.1,CCAFS SLC 40,SpaceX CRS-2 (Dragon C104),"4,877 kg (10,752 lb)",LEO (ISS),NASA (CRS),Success,No attempt,2013-03-01,4877.0,1,0,Other / no attempt,No attempt


,FlightNumber,DateText,BoosterVersion,LaunchSite,Payload,PayloadMassText,Orbit,Customer,LaunchOutcome,LandingOutcome,Date,PayloadMass,LaunchSuccess,LandingSuccess,LandingType,LandingResult
117,117.0,9 May 2021 06:42,F9 B5 ♺ B1051.10,CCSFS SLC 40,Starlink 27 v1.0 (60 satellites),"15,600 kg (34,400 lb)",LEO,SpaceX,Success,Success (drone ship),2021-05-09,15600.0,1,1,Drone ship,Success
118,118.0,15 May 2021 22:56,F9 B5 ♺ B1058.8,KSC LC 39A,Starlink 26 v1.0 (52 Satellites) Capella-6 &T...,"~14,000 kg (31,000 lb)",LEO,SpaceX Capella Space and Tyvak,Success,Success (drone ship),2021-05-15,14000.0,1,1,Drone ship,Success
119,119.0,26 May 2021 18:59,F9 B5 ♺ B1063.2,CCSFS SLC 40,Starlink 28 v1.0 (60 Satellites),"15,600 kg (34,400 lb)",LEO,SpaceX,Success,Success (drone ship),2021-05-26,15600.0,1,1,Drone ship,Success
120,120.0,3 June 2021 17:29,F9 B5 B1067.1,KSC LC 39A,SpaceX CRS-22 (Dragon C209.1),"3,328 kg (7,337 lb)",LEO (ISS),NASA (CRS),Success,Success (drone ship),2021-06-03,3328.0,1,1,Drone ship,Success
121,121.0,6 June 2021 04:26,F9 B5 ♺ B1061.3,CCSFS SLC 40,SXM-8,"7,000 kg (15,000 lb)",GTO,Sirius XM,Success,Success (drone ship),2021-06-06,7000.0,1,1,Drone ship,Success
